# SDM invullen

### Imports

In [31]:
import sqlite3
import pandas as pd
import os

### Verbinding maken met SDM

In [32]:
SDM = sqlite3.connect("../../Database/SDM2.db")
cursor = SDM.cursor()

dbs = {
    "Accessoireverkoop": sqlite3.connect("../../Database/BikeToDrive_1_Accessoireverkoop.db"),
    "Fietsverkoop": sqlite3.connect("../../Database/BikeToDrive_2_Fietsverkoop.db"),
    "Onderhoud": sqlite3.connect("../../Database/BikeToDrive_3_Onderhoud.db"),
    "Accessoire_Inkoop": sqlite3.connect("../../Database/BikeToDrive_4_Accessoire_Inkoop.db"),
    "Fiets_Inkoop": sqlite3.connect("../../Database/BikeToDrive_5_Fiets_Inkoop.db"),
}

### Bron databases openen

In [3]:
# def get_tables(conn):
#     return pd.read_sql_query(
#         "SELECT name FROM sqlite_master WHERE type='table';",
#         conn
#     )['name'].tolist()


### SDM schema dictionary

In [33]:
def get_tables(conn):
    return pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()


def load_db(conn, name):
    data = {}

    for t in get_tables(conn):
        df = pd.read_sql_query(f"SELECT * FROM {t}", conn)
        key = f"{name}_{t}"
        data[key] = df
        print(f"Loaded: {key}")

    return data


all_data = {}
for name, conn in dbs.items():
    all_data.update(load_db(conn, name))


Loaded: Accessoireverkoop_Accessoire_Verkoop
Loaded: Accessoireverkoop_Monteur
Loaded: Accessoireverkoop_Leverancier
Loaded: Accessoireverkoop_Accessoire
Loaded: Accessoireverkoop_Filiaal
Loaded: Accessoireverkoop_Klant
Loaded: Fietsverkoop_Fiets_Verkoop
Loaded: Fietsverkoop_Fiets
Loaded: Fietsverkoop_Monteur
Loaded: Fietsverkoop_Fabrikant
Loaded: Fietsverkoop_Filiaal
Loaded: Fietsverkoop_Klant
Loaded: Onderhoud_Onderhoud
Loaded: Onderhoud_Fiets
Loaded: Onderhoud_Monteur
Loaded: Onderhoud_Fabrikant
Loaded: Onderhoud_Filiaal
Loaded: Accessoire_Inkoop_Accessoire_Inkoop
Loaded: Accessoire_Inkoop_Accessoire
Loaded: Accessoire_Inkoop_Leverancier
Loaded: Fiets_Inkoop_Fiets_Inkoop
Loaded: Fiets_Inkoop_Fiets
Loaded: Fiets_Inkoop_Fabrikant


Dit zorgt er voor dat we later makelijker dingen kunnen doen zoals:
1. automatisch tabellen maken
2. automatisch tabellen leegmaken
3. automatisch data laden

### Reset_knop
a.	Alle tabellen uit het SDM leegmaken. Dit is dus een soort “reset-knop” voor de invulling van je SDM-tabellen.

In [34]:
def reset_sdm(conn):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    for t in reversed(tables):
        try:
            conn.execute(f"DELETE FROM {t}")
            print(f"Cleared: {t}")
        except:
            pass

    conn.commit()


reset_sdm(SDM)


Cleared: Fiets_Inkoop
Cleared: Fiets_Inkoop_Fiets
Cleared: Fiets_Inkoop_Fabrikant
Cleared: Accessoire_Inkoop
Cleared: Accessoire_Inkoop_Accessoire
Cleared: Accessoire_Inkoop_Leverancier
Cleared: Onderhoud
Cleared: Onderhoud_Monteur
Cleared: Onderhoud_Filiaal
Cleared: Onderhoud_Fiets
Cleared: Onderhoud_Fabrikant
Cleared: Fietsverkoop_Fiets_Verkoop
Cleared: Fietsverkoop_Fiets
Cleared: Fietsverkoop_Monteur
Cleared: Fietsverkoop_Klant
Cleared: Fietsverkoop_Filiaal
Cleared: Fietsverkoop_Fabrikant
Cleared: Accessoireverkoop_Accessoire_Verkoop
Cleared: Accessoireverkoop_Accessoire
Cleared: Accessoireverkoop_Klant
Cleared: Accessoireverkoop_Monteur
Cleared: Accessoireverkoop_Filiaal
Cleared: Accessoireverkoop_Leverancier


### Data uit bron lezen

In [35]:
SDM_MAPPING = {
    "Leverancier": [
        "Accessoireverkoop_Leverancier",
        "Accessoire_Inkoop_Leverancier"
    ],

    "Filiaal": [
        "Accessoireverkoop_Filiaal",
        "Fietsverkoop_Filiaal",
        "Onderhoud_Filiaal"
    ],

    "Fabrikant": [
        "Fietsverkoop_Fabrikant",
        "Onderhoud_Fabrikant",
        "Fiets_Inkoop_Fabrikant"
    ],

    "Klant": [
        "Accessoireverkoop_Klant",
        "Fietsverkoop_Klant"
    ],

    "Monteur": [
        "Accessoireverkoop_Monteur",
        "Fietsverkoop_Monteur",
        "Onderhoud_Monteur"
    ],

    "Accessoire": [
        "Accessoireverkoop_Accessoire",
        "Accessoire_Inkoop_Accessoire"
    ],

    "Fiets": [
        "Fietsverkoop_Fiets",
        "Onderhoud_Fiets",
        "Fiets_Inkoop_Fiets"
    ]
}


In [36]:
def insert(df, table, conn):
    df.to_sql(table, conn, if_exists="append", index=False)
    print(f"Inserted → {table}")


### Data overzetten naar SDM
b.	Data van elk .db-bestand overzetten naar het Source Data Model. Besteed hierbij extra aandacht aan de database-overschrijdende associaties.


In [37]:
def load_dimensions(mapping, data, conn):
    for entity, tables in mapping.items():
        for table in tables:

            source_key = None
            for k in data.keys():
                if k.endswith(f"_{entity}"):
                    source_key = k

            if source_key:
                insert(data[source_key], table, conn)
            else:
                print(f"Missing: {entity}")


load_dimensions(SDM_MAPPING, all_data, SDM)


Inserted → Accessoireverkoop_Leverancier
Inserted → Accessoire_Inkoop_Leverancier
Inserted → Accessoireverkoop_Filiaal
Inserted → Fietsverkoop_Filiaal
Inserted → Onderhoud_Filiaal
Inserted → Fietsverkoop_Fabrikant
Inserted → Onderhoud_Fabrikant
Inserted → Fiets_Inkoop_Fabrikant
Inserted → Accessoireverkoop_Klant
Inserted → Fietsverkoop_Klant
Inserted → Accessoireverkoop_Monteur
Inserted → Fietsverkoop_Monteur
Inserted → Onderhoud_Monteur
Inserted → Accessoireverkoop_Accessoire
Inserted → Accessoire_Inkoop_Accessoire
Inserted → Fietsverkoop_Fiets
Inserted → Onderhoud_Fiets
Inserted → Fiets_Inkoop_Fiets


In [ ]:
fact_tables = {
    "Onderhoud": "Onderhoud_Onderhoud",
    "Accessoire_Inkoop": "Accessoire_Inkoop_Accessoire_Inkoop",
    "Fiets_Inkoop": "Fiets_Inkoop_Fiets_Inkoop",
    "Accessoireverkoop_Accessoire_Verkoop": "Accessoireverkoop_Accessoire_Verkoop",
    "Fietsverkoop_Fiets_Verkoop": "Fietsverkoop_Fiets_Verkoop"
}

for target, source in fact_tables.items():
    if source in all_data:
        all_data[source].to_sql(target, SDM, if_exists="append", index=False)
        print(f"FACT LOADED → {target}")


FACT LOADED → Accessoireverkoop_Accessoire_Verkoop
FACT LOADED → Fietsverkoop_Fiets_Verkoop


In [39]:
SDM.commit()


In [40]:
def check_counts(conn):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    for t in tables:
        count = pd.read_sql_query(f"SELECT COUNT(*) as c FROM {t}", conn)['c'][0]
        print(f"{t}: {count}")


check_counts(SDM)


Accessoireverkoop_Leverancier: 5
Accessoireverkoop_Filiaal: 5
Accessoireverkoop_Monteur: 15
Accessoireverkoop_Klant: 25
Accessoireverkoop_Accessoire: 13
Accessoireverkoop_Accessoire_Verkoop: 100
Fietsverkoop_Fabrikant: 10
Fietsverkoop_Filiaal: 5
Fietsverkoop_Klant: 25
Fietsverkoop_Monteur: 15
Fietsverkoop_Fiets: 75
Fietsverkoop_Fiets_Verkoop: 150
Onderhoud_Fabrikant: 10
Onderhoud_Fiets: 75
Onderhoud_Filiaal: 5
Onderhoud_Monteur: 15
Onderhoud: 0
Accessoire_Inkoop_Leverancier: 5
Accessoire_Inkoop_Accessoire: 13
Accessoire_Inkoop: 0
Fiets_Inkoop_Fabrikant: 10
Fiets_Inkoop_Fiets: 75
Fiets_Inkoop: 0


### 6. testing
Test je Jupyter Notebook door in de operationele databronnen bepaalde wijzigingen aan te brengen (rijen toevoegen, updaten of verwijderen) en vervolgens te kijken of het SDM deze updates volledig overneemt na het uitvoeren van het Jupyter Notebook.

insert test


In [27]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    SDM
)

print(tables)


                                    name
0          Accessoireverkoop_Leverancier
1              Accessoireverkoop_Filiaal
2              Accessoireverkoop_Monteur
3                Accessoireverkoop_Klant
4           Accessoireverkoop_Accessoire
5   Accessoireverkoop_Accessoire_Verkoop
6                 Fietsverkoop_Fabrikant
7                   Fietsverkoop_Filiaal
8                     Fietsverkoop_Klant
9                   Fietsverkoop_Monteur
10                    Fietsverkoop_Fiets
11            Fietsverkoop_Fiets_Verkoop
12                   Onderhoud_Fabrikant
13                       Onderhoud_Fiets
14                     Onderhoud_Filiaal
15                     Onderhoud_Monteur
16                             Onderhoud
17         Accessoire_Inkoop_Leverancier
18          Accessoire_Inkoop_Accessoire
19                     Accessoire_Inkoop
20                Fiets_Inkoop_Fabrikant
21                    Fiets_Inkoop_Fiets
22                          Fiets_Inkoop


In [9]:
#leest alle rijen bron3/"Database/BikeToDrive_3_Onderhoud.db" => filiaal
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", bron3)

#resultaat
#replace omdat alles verwijder moet en de nieuwe data erin wordt gezet
df_filiaal.to_sql("Filiaal", db_path, if_exists="replace", index=False)

#data inlezen van filiaal => alleen rij met juiste id
df_check = pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr=123", db_path)

#verwachte uitkomst is dat er een nieuwe rij is toegevoegd met filiaalnr 123
print(df_check)

   filiaalnr          naam            adres provincie
0        123  Test Filiaal  appel straat 11  Den Haag


update test

In [30]:
def check_counts(conn):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    for t in tables:
        count = pd.read_sql_query(f"SELECT COUNT(*) as c FROM {t}", conn)['c'][0]
        print(f"{t}: {count} rows")

check_counts(SDM)


Accessoireverkoop_Leverancier: 5 rows
Accessoireverkoop_Filiaal: 5 rows
Accessoireverkoop_Monteur: 15 rows
Accessoireverkoop_Klant: 25 rows
Accessoireverkoop_Accessoire: 13 rows
Accessoireverkoop_Accessoire_Verkoop: 0 rows
Fietsverkoop_Fabrikant: 10 rows
Fietsverkoop_Filiaal: 5 rows
Fietsverkoop_Klant: 25 rows
Fietsverkoop_Monteur: 15 rows
Fietsverkoop_Fiets: 75 rows
Fietsverkoop_Fiets_Verkoop: 0 rows
Onderhoud_Fabrikant: 10 rows
Onderhoud_Fiets: 75 rows
Onderhoud_Filiaal: 5 rows
Onderhoud_Monteur: 15 rows
Onderhoud: 0 rows
Accessoire_Inkoop_Leverancier: 5 rows
Accessoire_Inkoop_Accessoire: 13 rows
Accessoire_Inkoop: 0 rows
Fiets_Inkoop_Fabrikant: 10 rows
Fiets_Inkoop_Fiets: 75 rows
Fiets_Inkoop: 0 rows


In [11]:
#leest alle rijen bron3/"Database/BikeToDrive_3_Onderhoud.db" => filiaal
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", bron3)

#resultaat
#replace omdat alles verwijder moet en de nieuwe data erin wordt gezet
df_filiaal.to_sql("Filiaal", db_path, if_exists="replace", index=False)

#data inlezen van filiaal => alleen rij met juiste id
df_check = pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr=123", db_path)

#verwachte uitkomst is dat de naam wordt veranderd 
print(df_check)

   filiaalnr         naam            adres provincie
0        123  New filiaal  appel straat 11  Den Haag


delete Test

In [12]:
#delete via cursor => rij verwijderd van filiaal
cursor_bron3.execute("""
DELETE FROM Filiaal
WHERE filiaalnr = 123
""")
bron3.commit()
print("Filiaal verwijderd")



Filiaal verwijderd


In [13]:
#leest alle rijen bron3/"Database/BikeToDrive_3_Onderhoud.db" => filiaal
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", bron3)

df_filiaal.to_sql("Filiaal", db_path, if_exists="replace", index=False)

7

In [14]:
df_check = pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr=123", db_path)

#verwachte uitkomst is dat de rij verwijderd is => index leeg
print(df_check)


Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
